# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List all RecordSets, their `@id`, fields, and columns
Entities are always referenced by their `@id`.

Let's enumerate all RecordSets and their details.

In [ ]:
# List all record sets and their details
record_sets = list(dataset.record_sets())

print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print(f"  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print("\n-----------------------------\n")

---
For further code examples, select a record set from above (e.g., the main protein table) and use its `@id`. See the output above for full `@id` values.

To examine a few records from a specific record set, use the `mlcroissant` API with the `record_set` argument set to that `@id`.

In [ ]:
# Example: print the first 2 records from the primary record set
# Please replace this ID with the primary table's @id from above.
main_record_set_id = None
for rs in dataset.record_sets():
    if 'protein' in rs.name.lower() or 'main' in rs.name.lower():
        main_record_set_id = rs.id
        break
if not main_record_set_id:
    # Fallback: just use the first one.
    main_record_set_id = record_sets[0].id

print(f"Primary record set `@id`: {main_record_set_id}\n\nSample records:")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    if i >= 2:
        break
    print(rec)
    print("-"*40)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

You can extract as many record sets as you need. For demonstration, we extract all record sets found in the metadata.

In [ ]:
# Extract data from each record set
df_dict = {}

for rs in record_sets:
    print(f"Loading record set: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        df_dict[rs.id] = df
        print(f"  Loaded {df.shape[0]} records, columns: {list(df.columns)}")
    else:
        print("  No records found in this record set.")

# For the following EDA, use the primary record set detected above
primary_rs_id = main_record_set_id
primary_df = df_dict[primary_rs_id]

print(f"\nPrimary record set ({primary_rs_id}) columns:")
print(primary_df.columns.tolist())
primary_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

- First, let's identify a numeric field (e.g., 'Molecular Weight', 'Coverage', 'PeptideCounts', or similar) for filtering and normalization.

In [ ]:
# Try to identify a numeric field from the DataFrame columns
import numpy as np

numeric_candidates = []
for col in primary_df.columns:
    series = pd.to_numeric(primary_df[col], errors='coerce')
    if series.notnull().sum() > 0 and np.issubdtype(series.dropna().dtype, np.number):
        numeric_candidates.append(col)

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # fallback to a common field name
    numeric_field = 'Molecular Weight' if 'Molecular Weight' in primary_df.columns else primary_df.columns[0]

print(f"Selected numeric field for EDA: {numeric_field}")

# Threshold (median as an example)
nums = pd.to_numeric(primary_df[numeric_field], errors='coerce')
threshold = nums.median() if not np.isnan(nums.median()) else 10

filtered_df = primary_df[pd.to_numeric(primary_df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f'{numeric_field}_normalized'] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - nums.mean()
) / nums.std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

# Find a grouping/categorical field
group_field = None
for col in primary_df.columns:
    if col != numeric_field and primary_df[col].nunique() < 20:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Simple histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(primary_df[numeric_field], errors='coerce').dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping field available, boxplot
if group_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(
        x=primary_df[group_field],
        y=pd.to_numeric(primary_df[numeric_field], errors='coerce')
    )
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using its Croissant schema and the `mlcroissant` Python library,
- Explored metadata to identify record sets, fields, and columns using their `@id`,
- Loaded tabular data into Pandas DataFrames for analysis,
- Performed basic exploratory data analysis, filtering, normalization, and grouped aggregation on the main protein record set,
- Visualized distributions and relationships between fields.

Further analyses may include more advanced filtering, merging additional related record sets by entity ID, or integrating with scientific protein databases for annotation. For more details and API documentation, see [`mlcroissant`](https://mlcommons.github.io/croissant/api/).
